
Import Libraries

In [1]:
!pip install --upgrade neurokit2


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import pandas as pd
import warnings
import random
import matplotlib.pyplot as plt

import neurokit2 as nk
from scipy.stats.mstats import winsorize
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

Step 1 - Load CSV files


In [3]:
DATA_DIR = r"C:/Users/seanl/Desktop/DREAMER_EXPERIMENT"

df_eeg = pd.read_csv(os.path.join(DATA_DIR, "processed_dreamer_data_eeg_features.csv"))
df_ecg = pd.read_csv(os.path.join(DATA_DIR, "processed_dreamer_data_ecg_features.csv"))
df_fused = pd.read_csv(os.path.join(DATA_DIR, "processed_dreamer_data_fused_features.csv"))

print('EEG   shape:', df_eeg.shape)
print('ECG   shape:', df_ecg.shape)
print('Fused shape:', df_fused.shape)
print('\nValence distribution (all 5 levels):')
print(df_eeg['valence'].value_counts().sort_index())

EEG   shape: (412, 47)
ECG   shape: (412, 163)
Fused shape: (412, 205)

Valence distribution (all 5 levels):
valence
1.0     76
2.0     84
3.0     90
4.0    107
5.0     55
Name: count, dtype: int64


Step 2 - Define binary emotion model

In [4]:
LABEL_COLS = ['valence', 'arousal', 'dominance', 'video ID', 'participant']
TARGET = 'valence'
THRESHOLD = 4   # Paper: Happy >= 4,  Not Happy < 4

def create_binary_labels(df):
    return (df[TARGET].values >= THRESHOLD).astype(np.int32)

for name, df in [('EEG', df_eeg), ('ECG', df_ecg), ('Fused', df_fused)]:
    y = create_binary_labels(df)
    print(f'{name}: Happy (>=4) = {y.sum()}  |  Not Happy (<4) = {(y==0).sum()}')

EEG: Happy (>=4) = 162  |  Not Happy (<4) = 250
ECG: Happy (>=4) = 162  |  Not Happy (<4) = 250
Fused: Happy (>=4) = 162  |  Not Happy (<4) = 250


Step 3 - Normalise per participant 
Applies inside the LOSO loop so training and test are always normalised separately.

In [5]:
def normalise_per_participant(X_tr_df, X_te_df):
    """
    Per-participant normalisation:
      1. Robust standardise (Median / MAD) per participant group
      2. Winsorise at 5% each tail
      3. Rescale to [-4, 4]
    Fitted on training participants. Test participant is normalised
    using the overall training median/MAD to prevent leakage.
    """
    feat_cols = [c for c in X_tr_df.columns if c != 'participant']

    def norm_group(group):
        num = group[feat_cols].copy()
        std = nk.standardize(num, robust=True)
        wins = std.apply(lambda x: winsorize(x, limits=[0.05, 0.05]))
        rescaled = nk.rescale(wins, to=[-4, 4])
        group = group.copy()
        group[feat_cols] = rescaled
        return group

    # Normalise each training participant separately
    X_tr_norm = X_tr_df.groupby('participant', group_keys=False).apply(norm_group)

    # Use training median/MAD for the held-out test participant
    tr_vals    = X_tr_norm[feat_cols]
    tr_median  = tr_vals.median()
    tr_mad     = (tr_vals - tr_median).abs().median().replace(0, 1)

    te_std  = (X_te_df[feat_cols] - tr_median) / tr_mad
    te_wins = te_std.apply(lambda x: winsorize(x, limits=[0.05, 0.05]))
    te_resc = nk.rescale(te_wins, to=[-4, 4])

    X_tr_arr = X_tr_norm[feat_cols].values.astype(np.float32)
    X_te_arr = te_resc.values.astype(np.float32)

    X_tr_arr = np.nan_to_num(X_tr_arr, nan=0.0, posinf=4.0, neginf=-4.0)
    X_te_arr = np.nan_to_num(X_te_arr, nan=0.0, posinf=4.0, neginf=-4.0)

    return X_tr_arr, X_te_arr

print('Per-participant normalisation defined.')

Per-participant normalisation defined.


Step 4 - Subject-independent split (LOSO)

23 folds - one per participant. Each fold: that participant = test, remaining 22 = train.

In [6]:
def loso_folds(df):
    feat_cols = [c for c in df.columns if c not in ['valence', 'arousal', 'dominance', 'video ID']]
    y_all = create_binary_labels(df)
    participants = df['participant'].values
    X_df = df[feat_cols].reset_index(drop=True)

    logo = LeaveOneGroupOut()
    for train_idx, test_idx in logo.split(X_df, y_all, participants):
        pid = participants[test_idx[0]]
        yield (X_df.iloc[train_idx].reset_index(drop=True),
               X_df.iloc[test_idx].reset_index(drop=True),
               y_all[train_idx],
               y_all[test_idx],
               pid)

print('LOSO fold generator defined.')

LOSO fold generator defined.


Step 5 — Balance training set with SMOTE

In [7]:
def balance_smote(X, y):
    classes, counts = np.unique(y, return_counts=True)
    if len(classes) < 2 or min(counts) < 2:
        return X, y
    k = min(5, min(counts) - 1)
    sm = SMOTE(random_state=42, k_neighbors=k)
    return sm.fit_resample(X, y)

print('SMOTE defined.')

SMOTE defined.


Step 6 — Model definitions

In [ ]:
def build_lstm(input_dim):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=(1, input_dim)),
        Dropout(0.3),
        LSTM(64),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

MODEL_BUILDERS = {'LSTM': build_lstm}
print('Models defined: LSTM')

Models defined: LSTM


: 

Step 7 — Train and test 

In [9]:
def run_experiment(df, dataset_name, model_name):
    print(f'\n{"="*60}')
    print(f'  {model_name} | {dataset_name}')
    print(f'{"="*60}')

    accs, precs, recs, f1s, aucs = [], [], [], [], []
    es = EarlyStopping(patience=5, restore_best_weights=True, verbose=0)

    for X_tr_df, X_te_df, y_tr, y_te, pid in loso_folds(df):

        # Step 3: per-participant normalisation, no leakage
        X_tr, X_te = normalise_per_participant(X_tr_df, X_te_df)

        # Step 5: balance with SMOTE
        X_tr_bal, y_tr_bal = balance_smote(X_tr, y_tr)

        # Reshape to (samples, 1 timestep, features)
        X_tr_3d = X_tr_bal.reshape(X_tr_bal.shape[0], 1, X_tr_bal.shape[1])
        X_te_3d = X_te.reshape(X_te.shape[0],         1, X_te.shape[1])

        # Step 6: train
        model = MODEL_BUILDERS[model_name](X_tr_3d.shape[2])
        model.fit(X_tr_3d, y_tr_bal,
                  epochs=50, batch_size=32,
                  validation_split=0.1,
                  callbacks=[es], verbose=0)

        # Step 7: test
        y_prob = model.predict(X_te_3d, verbose=0).ravel()
        y_pred = (y_prob >= 0.5).astype(int)

        if len(np.unique(y_te)) < 2:
            continue

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred, zero_division=0))
        recs.append(recall_score(y_te, y_pred, zero_division=0))
        f1s.append(f1_score(y_te, y_pred, zero_division=0))
        aucs.append(roc_auc_score(y_te, y_prob))

    results = {
        'Model': model_name,
        'Dataset': dataset_name,
        'Accuracy': np.mean(accs),
        'Precision': np.mean(precs),
        'Recall': np.mean(recs),
        'F1': np.mean(f1s),
        'AUC': np.mean(aucs)
    }
    print(f'  Acc={results["Accuracy"]:.3f}  Prec={results["Precision"]:.3f}  '
          f'Rec={results["Recall"]:.3f}  F1={results["F1"]:.3f}  AUC={results["AUC"]:.3f}')
    return results

print('run_experiment() defined.')

run_experiment() defined.


Run all experiments

In [ ]:
DATASETS = [
    ('EEG', df_eeg),
    ('ECG', df_ecg),
    ('Fused', df_fused)
]

all_results = []
for model_name in ['LSTM']:
    for ds_name, df in DATASETS:
        res = run_experiment(df, ds_name, model_name)
        all_results.append(res)



  LSTM | EEG
  Acc=0.592  Prec=0.019  Rec=0.038  F1=0.025  AUC=0.460

  LSTM | ECG
  Acc=0.602  Prec=0.019  Rec=0.043  F1=0.027  AUC=0.595

  LSTM | Fused
  Acc=0.607  Prec=0.000  Rec=0.000  F1=0.000  AUC=0.520

All done!


Results Summary Table

In [11]:
results_df = pd.DataFrame(all_results).round(3)
results_df = results_df.sort_values(['Model', 'Dataset']).reset_index(drop=True)
print(results_df.to_string(index=False))

out_path = os.path.join(DATA_DIR, 'LSTM_model_results_summary.csv')
results_df.to_csv(out_path, index=False)
print(f'\nSaved to: {out_path}')

Model Dataset  Accuracy  Precision  Recall    F1   AUC
 LSTM     ECG     0.602      0.019   0.043 0.027 0.595
 LSTM     EEG     0.592      0.019   0.038 0.025 0.460
 LSTM   Fused     0.607      0.000   0.000 0.000 0.520

Saved to: C:/Users/seanl/Desktop/DREAMER_EXPERIMENT\LSTM_model_results_summary.csv
